# HLA Inference from TCRαβ Repertoires — Rosati 2022 Cohort

**Dataset**: Rosati et al. 2022 (PRJEB50045) — bulk TCRαβ sequencing from 269 individuals across four clinical groups: Healthy controls, Crohn's Disease (CD), Ulcerative Colitis (UC), and Colorectal Cancer (CRC).

**Objective**: Apply HLAGuessr (Ruiz Ortega et al. 2025) to infer HLA genotypes from TCR repertoires. **Important caveat**: Rosati's cohort was one of three datasets used in HLAGuessr's original training (alongside Russell et al. and Emerson et al.), with HLA genotypes obtained from the authors for that purpose. Predictions on this cohort are therefore not fully blind — high-confidence results are *expected by design* rather than a test of generalisation. This inference step was performed to later confirm that expectation once ground-truth HLA became available to us directly (see `rosati_qc_hla_validation.ipynb`).

**Pipeline overview**:
1. Download raw FASTQs from ENA (PRJEB50045)
2. Process with MiXCR (milab-human-rna-tcr-umi-multiplex preset)
3. Consolidate clonotype tables → processed CSV
4. Quality control by repertoire depth
5. HLA inference with HLAGuessr (94 modelable alleles)
6. Export predicted genotypes per patient

**Reference**: Ruiz Ortega et al. 2025, *PLOS Computational Biology*, DOI: 10.1371/journal.pcbi.1012724

## 0. Imports & paths

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import time
import warnings
import matplotlib.pyplot as plt
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from HLAGuessr.processing import Processing
from HLAGuessr.infer_hla import GetProbabilities

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE          = '/home/immunologylab/bioinformatics/'
FASTQ_DIR     = BASE + 'raw_data/rosati_fastq'
UMI_DIR       = BASE + 'raw_data/rosati_umi'
MIXCR_DIR     = BASE + 'raw_data/rosati_mixcr'
MIXCR_BIN     = BASE + 'mixcr'
SRA_CSV       = BASE + 'rosati_sra.csv'
PROCESSED_CSV = BASE + 'hla-tcr-project/data/processed/rosati_repertoires_processed.csv'
RESULTS_DIR   = BASE + 'hla-tcr-project/results/'
HLAGUESSR_OUT = RESULTS_DIR + 'hlaguessr_rosati/'
CROSSVAL_CSV  = RESULTS_DIR + 'hlaguessr_46B_crossval.csv'

## 1. Data ingestion — SRA metadata

The SRA RunInfo table (downloaded from NCBI) maps each sequencing run to a patient and library type. Each patient has multiple runs (Alpha, Beta, or AlphaBeta libraries). Patient IDs are extracted from the `LibraryName` field (format: `1_CD_1_Alpha.v2` → `CD_1`).

In [ ]:
sra = pd.read_csv(SRA_CSV)

sra['patient_id'] = sra['LibraryName'].apply(
    lambda x: '_'.join(x.split('_')[1:3]) if pd.notna(x) else ''
)
sra['chain'] = sra['LibraryName'].apply(
    lambda x: 'Alpha' if 'Alpha' in x and 'Beta' not in x
    else ('Beta' if 'Beta' in x and 'Alpha' not in x else 'AlphaBeta')
)

print(f'Total runs:      {len(sra)}')
print(f'Unique patients: {sra["patient_id"].nunique()}')
print(f'Runs by type:    {sra["chain"].value_counts().to_dict()}')

## 2. MiXCR processing

Raw paired-end FASTQs are processed with MiXCR v4.7 using the `milab-human-rna-tcr-umi-multiplex` preset (multiplex TCR amplicon with UMI correction). UMI sequences are prepended to R1 before alignment. To maximise throughput, all 1,448 runs are processed in parallel (14 simultaneous jobs, 2 threads each) using GNU parallel.

The two cells below generate: (1) a standalone Python worker script for a single run, and (2) a TSV argument list consumed by `parallel`.

In [ ]:
N_JOBS = 14  # server has 28 cores; MiXCR uses 2 threads internally per run

process_script = '''#!/usr/bin/env python3
import gzip, re, subprocess, sys, os

def prepend_umi(r1_in, r2_in, r1_out, r2_out):
    """Prepend UMI from MIGUMI tag in R1 header to the read sequence."""
    with gzip.open(r1_in, "rt") as f1, gzip.open(r2_in, "rt") as f2, \\
         gzip.open(r1_out, "wt") as o1, gzip.open(r2_out, "wt") as o2:
        while True:
            h1 = f1.readline()
            if not h1: break
            s1=f1.readline().strip(); p1=f1.readline().strip(); q1=f1.readline().strip()
            h2=f2.readline(); s2=f2.readline().strip(); p2=f2.readline().strip(); q2=f2.readline().strip()
            m = re.search(r"MIGUMI:([ACGTN]+)", h1)
            umi = m.group(1) if m else ""
            o1.write(h1); o1.write(umi+s1+"\\n"); o1.write(p1+"\\n"); o1.write("I"*len(umi)+q1+"\\n")
            o2.write(h2); o2.write(s2+"\\n"); o2.write(p2+"\\n"); o2.write(q2+"\\n")

run, r1, r2, r1u, r2u, out, mixcr = sys.argv[1:]

if os.path.exists(out + ".clns"):
    print(f"SKIP {run}")
    sys.exit(0)

prepend_umi(r1, r2, r1u, r2u)
result = subprocess.run([
    mixcr, "analyze", "milab-human-rna-tcr-umi-multiplex",
    "--species", "hsa", "--tag-pattern", "^(UMI:N{12})(R1:*)\\\\^(R2:*)",
    "--threads", "2", r1u, r2u, out
], capture_output=True, text=True)
os.remove(r1u); os.remove(r2u)
print(f"OK {run}" if result.returncode == 0 else f"ERROR {run}: {result.stderr[-200:]}")
'''

with open(BASE + 'process_single_run.py', 'w') as f:
    f.write(process_script)

args_lines = []
for _, row in sra.iterrows():
    run = row['Run']
    args_lines.append('\t'.join([
        run,
        f'{FASTQ_DIR}/{run}_1.fastq.gz', f'{FASTQ_DIR}/{run}_2.fastq.gz',
        f'{UMI_DIR}/{run}_1_umi.fastq.gz', f'{UMI_DIR}/{run}_2_umi.fastq.gz',
        f'{MIXCR_DIR}/{run}', MIXCR_BIN
    ]))

args_file = BASE + 'rosati_runs_args.tsv'
with open(args_file, 'w') as f:
    f.write('\n'.join(args_lines))

print(f'Worker script : {BASE}process_single_run.py')
print(f'Arguments file: {args_file}')
print(f'Total runs    : {len(args_lines)}')
print(f'Parallel jobs : {N_JOBS}')
print(f'Est. runtime  : ~{len(args_lines) * 30 / N_JOBS / 60:.0f} min')
print()
print('Run with: parallel -j 14 python3 process_single_run.py :::: rosati_runs_args.tsv')

## 3. Post-processing — verify completeness & consolidate

After MiXCR finishes, we verify how many patients have both TRA and TRB clonotype files, then consolidate all per-run TSVs into a single processed CSV. CDR3 sequences are filtered to canonical amino acid boundaries (`^C[A-Z]+F$` or `^C[A-Z]+[WY]$`). V-gene assignments are taken from the top-scoring hit in `allVHitsWithScore`.

In [ ]:
patients_with_alpha, patients_with_beta = set(), set()

for _, row in sra.iterrows():
    run, pid = row['Run'], row['patient_id']
    if os.path.exists(os.path.join(MIXCR_DIR, f'{run}.clones_TRA.tsv')): patients_with_alpha.add(pid)
    if os.path.exists(os.path.join(MIXCR_DIR, f'{run}.clones_TRB.tsv')): patients_with_beta.add(pid)

both = patients_with_alpha & patients_with_beta
print(f'Patients with TRA + TRB : {len(both)}')
print(f'Patients with TRA only  : {len(patients_with_alpha - patients_with_beta)}')
print(f'Patients with TRB only  : {len(patients_with_beta  - patients_with_alpha)}')
print(f'Total with any data     : {len(patients_with_alpha | patients_with_beta)}')

In [ ]:
def parse_vgene(v_hit_str):
    """Extract the top-scoring V gene from MiXCR's allVHitsWithScore field."""
    if pd.isna(v_hit_str) or v_hit_str == '':
        return None
    m = re.match(r'(TR[AB][VDJ]\d+(?:-\d+)?)', str(v_hit_str).split(',')[0].strip())
    return m.group(1) if m else None

print('Consolidating repertoires...')
frames = []

for patient_id in sorted(sra['patient_id'].unique()):
    for run in sra[sra['patient_id'] == patient_id]['Run'].tolist():
        for chain, suffix in [('TRA', 'clones_TRA.tsv'), ('TRB', 'clones_TRB.tsv')]:
            fpath = os.path.join(MIXCR_DIR, f'{run}.{suffix}')
            if not os.path.exists(fpath):
                continue
            try:
                df = pd.read_csv(fpath, sep='\t',
                                 usecols=['readCount', 'aaSeqCDR3', 'allVHitsWithScore'])
                if len(df) == 0: continue
                df['v_gene']     = df['allVHitsWithScore'].apply(parse_vgene)
                df['chain']      = chain
                df['patient_id'] = patient_id
                df['run']        = run
                df = df[df['aaSeqCDR3'].notna()]
                df = df[df['aaSeqCDR3'].str.match(r'^C[A-Z]+F$|^C[A-Z]+[WY]$')]
                frames.append(df[['patient_id', 'chain', 'aaSeqCDR3', 'v_gene', 'readCount', 'run']])
            except Exception as e:
                print(f'  Error {run} {chain}: {e}')

rosati = pd.concat(frames, ignore_index=True)
rosati.columns = ['patient_id', 'chain', 'cdr3aa', 'v_gene', 'read_count', 'run']
rosati.to_csv(PROCESSED_CSV, index=False)

print(f'Total clones : {len(rosati):,}')
print(f'Patients     : {rosati["patient_id"].nunique()}')
print(f'Saved to     : {PROCESSED_CSV}')
print()
print(rosati.groupby(['patient_id','chain']).size().unstack().describe().round(0))

## 4. Dataset overview & quality control

The cohort contains four clinical groups. CRC samples show extremely low clone counts (median 8 per patient) due to low T-cell infiltration in tumor tissue or failed sequencing runs — these are effectively uninformative for HLA inference. A minimum threshold of **1,000 clones per chain** (TRA and TRB) is applied to retain only repertoires with sufficient depth for HLAGuessr to find HLA-associated public clonotypes.

In [ ]:
rosati = pd.read_csv(PROCESSED_CSV)

def get_group(pid):
    if pid.startswith('CRC_'):      return 'CRC'
    elif pid.startswith('CD_'):     return 'CD'
    elif pid.startswith('UC_'):     return 'UC'
    elif pid.startswith('Healthy'): return 'Healthy'
    else: return 'Other'

rosati['group'] = rosati['patient_id'].apply(get_group)
clones = rosati.groupby(['patient_id','group','chain']).size().unstack(fill_value=0)
clones.columns = ['n_TRA','n_TRB']
clones = clones.reset_index()

# Summary table
groups  = ['Healthy','CD','UC','CRC']
colors  = {'Healthy':'#1D9E75','CD':'#3A86FF','UC':'#FFBE0B','CRC':'#E24B4A'}
stats   = clones.groupby('group').agg(
    N=('patient_id','count'),
    TRA_median=('n_TRA','median'), TRA_mean=('n_TRA','mean'),
    TRA_min=('n_TRA','min'),       TRA_max=('n_TRA','max'),
    TRB_median=('n_TRB','median'), TRB_mean=('n_TRB','mean'),
    TRB_min=('n_TRB','min'),       TRB_max=('n_TRB','max'),
).reindex(groups).round(0).astype(int)

print('Clones per patient by clinical group')
print('='*100)
print(f"{'Group':<10} {'N':>4} {'TRA median':>12} {'TRA mean':>10} {'TRA min':>8} {'TRA max':>10} {'TRB median':>12} {'TRB mean':>10} {'TRB min':>8} {'TRB max':>10}")
print('-'*100)
for grp in groups:
    r = stats.loc[grp]
    print(f"{grp:<10} {r['N']:>4} {r['TRA_median']:>12,} {r['TRA_mean']:>10,} {r['TRA_min']:>8,} {r['TRA_max']:>10,} {r['TRB_median']:>12,} {r['TRB_mean']:>10,} {r['TRB_min']:>8,} {r['TRB_max']:>10,}")
print('='*100)

# Bar chart: mean ± SD per group
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, label in zip(axes, ['n_TRA','n_TRB'], ['TCRα','TCRβ']):
    means   = [clones[clones['group']==g][col].mean()   for g in groups]
    medians = [clones[clones['group']==g][col].median() for g in groups]
    stds    = [clones[clones['group']==g][col].std()    for g in groups]
    ns      = [clones[clones['group']==g][col].count()  for g in groups]
    x = np.arange(len(groups))
    ax.bar(x, means, color=[colors[g] for g in groups], alpha=0.85, edgecolor='white')
    ax.errorbar(x, means, yerr=stds, fmt='none', color='black', capsize=4, linewidth=1.2)
    for i, (med, n) in enumerate(zip(medians, ns)):
        ax.hlines(med, i-0.35, i+0.35, colors='black', linewidth=2, linestyle='--')
        ax.text(i, -5000, f'n={n}', ha='center', fontsize=9,
                color=colors[groups[i]], fontweight='bold')
        ax.text(i, med+500, f'med={med:,.0f}', ha='center', fontsize=7.5, color='black', fontweight='bold')
    ax.axhline(1000, color='red', linestyle='--', alpha=0.5, linewidth=1, label='QC threshold (1,000)')
    ax.set_xticks(x); ax.set_xticklabels(groups, fontsize=11)
    ax.set_ylabel('Clones per patient', fontsize=10)
    ax.set_title(f'{label} — Mean ± SD', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_ylim(-7000, max(means)+max(stds)+5000)

plt.suptitle('Rosati 2022 — Repertoire depth by clinical group', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR + 'rosati_qc_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
MIN_CLONES = 1000
clones['pass_qc'] = (clones['n_TRA'] >= MIN_CLONES) & (clones['n_TRB'] >= MIN_CLONES)

print(f'QC filter: ≥{MIN_CLONES} clones per chain (TRA and TRB)')
print()
for grp in groups:
    n_pass  = clones[(clones['group']==grp) & clones['pass_qc']].shape[0]
    n_total = clones[clones['group']==grp].shape[0]
    print(f'  {grp:<10}: {n_pass}/{n_total} pass QC')

valid_patients = clones[clones['pass_qc']]['patient_id'].tolist()
print(f'\nTotal patients passing QC: {len(valid_patients)}')

## 5. HLA inference with HLAGuessr

HLAGuessr infers HLA alleles by searching for HLA-associated public TCR clonotypes in each repertoire. It runs a binary L1-regularised logistic classifier independently per allele (`untyped=True` mode — no ground-truth required). We use the 94 alleles that were modelable in the 46_B cross-validation (i.e. alleles with sufficient carriers in the HLAGuessr training set). Input files per patient use V-gene family notation (e.g. `TRBV6`, not `TRBV6-5`) to match the training index.

In [ ]:
def norm_vfamily(v):
    """Collapse IMGT V-gene to family level: TRBV6-5 → TRBV6."""
    if pd.isna(v): return None
    m = re.match(r'(TR[AB]V\d+)', str(v))
    return m.group(1) if m else None

rosati['v_family'] = rosati['v_gene'].apply(norm_vfamily)

alpha = rosati[rosati['chain']=='TRA'][['cdr3aa','v_family','patient_id']]\
        .rename(columns={'patient_id':'Patient'}).dropna()
beta  = rosati[rosati['chain']=='TRB'][['cdr3aa','v_family','patient_id']]\
        .rename(columns={'patient_id':'Patient'}).dropna()

os.makedirs(HLAGUESSR_OUT, exist_ok=True)

for pat in valid_patients:
    pat_dir = HLAGUESSR_OUT + pat + '/'
    os.makedirs(pat_dir, exist_ok=True)
    alpha[alpha['Patient']==pat].to_csv(pat_dir + 'alpha.tsv', sep='\t', index=False)
    beta[beta['Patient']==pat].to_csv(pat_dir  + 'beta.tsv',  sep='\t', index=False)

print(f'Input folders created: {len(valid_patients)}')
print(f'Alpha clones: {len(alpha):,} | Beta clones: {len(beta):,}')

In [ ]:
# Allele list: the 94 modelable alleles from 46_B cross-validation
cv_df   = pd.read_csv(CROSSVAL_CSV)
alleles = sorted(cv_df['HLA'].unique().tolist())

chain, delimiter, grouping = ['alpha', 'beta'], '\t', True

print(f'Patients : {len(valid_patients)}')
print(f'Alleles  : {len(alleles)}')
print(f'Total predictions: {len(valid_patients) * len(alleles):,}')

results = []
t0      = time.time()

for i, pat in enumerate(valid_patients):
    pat_dir  = HLAGUESSR_OUT + pat + '/'
    out_file = pat_dir + 'predictions.csv'

    # Resume if already done
    if os.path.exists(out_file):
        results.append(pd.read_csv(out_file))
        continue

    try:
        dm = Processing(chain, pat_dir+'alpha.tsv', pat_dir+'beta.tsv', delimiter, grouping)
    except Exception as e:
        print(f'ERROR loading {pat}: {e}')
        continue

    pat_results = []
    for hla_target in alleles:
        try:
            gp = GetProbabilities(
                hla_target=hla_target, chain=chain,
                ps=dm.ps, data_test=dm.data_test,
                grouping=True, untyped=True
            )
            pat_results.append({
                'patient_id': pat, 'HLA': hla_target,
                'probability': gp.hla_prob, 'predicted_carrier': gp.hla_output
            })
        except Exception:
            pass

    if pat_results:
        df_pat = pd.DataFrame(pat_results)
        df_pat.to_csv(out_file, index=False)
        results.append(df_pat)

    elapsed = time.time() - t0
    eta     = elapsed / (i+1) * (len(valid_patients) - i - 1)
    print(f'{i+1}/{len(valid_patients)} {pat} | {elapsed/60:.1f} min | ETA {eta/60:.1f} min')

predictions = pd.concat(results, ignore_index=True)
predictions.to_csv(RESULTS_DIR + 'hlaguessr_rosati_predictions.csv', index=False)
print(f'\nDone in {(time.time()-t0)/60:.1f} min | {len(predictions):,} predictions saved')

## 6. Results — predicted carriers per allele

When a prediction is available (non-NaN probability), confidence is consistently high (mean >0.98 across all groups), reflecting the logistic classifier's tendency to output near-binary probabilities. NaN values indicate that no HLA-associated public clonotypes for that allele were detected in the repertoire — this affects 19/233 QC-passing patients (8.2%) and is unrelated to repertoire size.

In [ ]:
predictions = pd.read_csv(RESULTS_DIR + 'hlaguessr_rosati_predictions.csv')
predictions['group'] = predictions['patient_id'].apply(get_group)

# NaN rate and confidence by group
print('=== SIGNAL COVERAGE BY GROUP ===')
for grp in ['Healthy','CD','UC','CRC']:
    sub = predictions[predictions['group']==grp]
    nan_pct  = sub['probability'].isna().mean() * 100
    carriers = sub[sub['predicted_carrier']==True]
    mean_prob = carriers['probability'].mean() if len(carriers) else float('nan')
    print(f'  {grp:<10}: NaN {nan_pct:.1f}% | mean prob when True: {mean_prob:.3f} | n carriers: {len(carriers)}')

print()
print('=== TOP 20 ALLELES BY PREDICTED CARRIER FREQUENCY ===')
carriers_all = predictions[predictions['predicted_carrier']==True]
top = carriers_all.groupby('HLA').agg(
    n_carriers=('patient_id','count'),
    mean_prob=('probability','mean')
).sort_values('n_carriers', ascending=False).head(20)
top['freq'] = (top['n_carriers'] / predictions['patient_id'].nunique() * 100).round(1)
top['mean_prob'] = top['mean_prob'].round(3)
print(top.to_string())

## 7. Predicted HLA genotype table

For each patient, the two highest-probability alleles per locus are reported — consistent with the diploid human genome (one allele per chromosome). Probabilities are shown alongside each call. Patients with no signal across any locus are marked `[NO SIGNAL]`. Results are exported to a formatted Excel workbook with separate sheets for Class I (A, B, C) and Class II (DRB1, DQB1, DQA1, DPB1).

In [ ]:
def get_locus(allele):
    if allele.startswith('DRB1'):   return 'DRB1'
    elif allele.startswith('DQB1'): return 'DQB1'
    elif allele.startswith('DQA1'): return 'DQA1'
    elif allele.startswith('DPB1'): return 'DPB1'
    else: return allele[0]

predictions['locus']    = predictions['HLA'].apply(get_locus)
predictions['prob_pct'] = (predictions['probability'] * 100).round(1)

carriers = predictions[predictions['predicted_carrier']==True]\
           [['patient_id','group','locus','HLA','prob_pct']]\
           .sort_values(['patient_id','locus','prob_pct'], ascending=[True,True,False])

loci_all = ['A','B','C','DRB1','DQB1','DQA1','DPB1']
rows = []
for pat in sorted(valid_patients):
    pat_df  = carriers[carriers['patient_id']==pat]
    row     = {'Patient': pat, 'Group': get_group(pat)}
    n_total = 0
    for locus in loci_all:
        sub = pat_df[pat_df['locus']==locus].sort_values('prob_pct', ascending=False).head(2)
        for k in [1, 2]:
            if len(sub) >= k:
                row[f'{locus}_{k}']      = sub.iloc[k-1]['HLA']
                row[f'{locus}_{k}_prob'] = f"{sub.iloc[k-1]['prob_pct']}%"
            else:
                row[f'{locus}_{k}']      = '—'
                row[f'{locus}_{k}_prob'] = '—'
        n_total += len(pat_df[pat_df['locus']==locus])
    row['N'] = n_total
    rows.append(row)

genotype = pd.DataFrame(rows)

pd.set_option('display.max_colwidth', 15)
pd.set_option('display.width', 400)

for grp in ['Healthy','CD','UC','CRC']:
    sub = genotype[genotype['Group']==grp]
    if len(sub) == 0: continue
    print(f'\n{"="*120}')
    print(f'  {grp} — {len(sub)} patients | mean {sub["N"].mean():.1f} alleles/patient | {(sub["N"]==0).sum()} with no signal')
    print(f'{"="*120}')
    cols_I  = ['Patient'] + [c for l in ['A','B','C'] for c in [f'{l}_1',f'{l}_1_prob',f'{l}_2',f'{l}_2_prob']]
    cols_II = ['Patient'] + [c for l in ['DRB1','DQB1','DQA1','DPB1'] for c in [f'{l}_1',f'{l}_1_prob',f'{l}_2',f'{l}_2_prob']]
    print('  CLASS I:')
    print(sub[cols_I].to_string(index=False))
    print('  CLASS II:')
    print(sub[cols_II].to_string(index=False))

print(f'\nTOTAL: {len(genotype)} patients | mean {genotype["N"].mean():.1f} alleles/patient | {(genotype["N"]==0).sum()} with no signal')

## 8. Export to Excel

The predicted genotype table is exported as a formatted `.xlsx` workbook with three sheets: Class I alleles, Class II alleles, and a summary page. Rows are colour-coded by clinical group. Patients with no signal are rendered in grey italic.

In [ ]:
group_colors = {'Healthy':'D4F5E9','CD':'D4E5FF','UC':'FFF8D4','CRC':'FFD4D4'}
HEADER_COLOR = '2D2D2D'
thin = Side(style='thin', color='CCCCCC')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

def write_row(ws, row_num, data, fill_color, grey_italic=False):
    for col, val in enumerate(data, 1):
        cell = ws.cell(row=row_num, column=col, value=val)
        cell.font = Font(name='Arial', size=9, italic=grey_italic,
                         color='999999' if grey_italic else '000000')
        cell.fill = PatternFill('solid', start_color=fill_color)
        cell.alignment = Alignment(horizontal='center', vertical='center')
        cell.border = border

def make_header_row(ws, headers):
    for col, h in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col, value=h)
        cell.font = Font(bold=True, color='FFFFFF', size=10, name='Arial')
        cell.fill = PatternFill('solid', start_color=HEADER_COLOR)
        cell.alignment = Alignment(horizontal='center', vertical='center')
        cell.border = border
    ws.row_dimensions[1].height = 30
    ws.freeze_panes = 'A2'

wb = Workbook()

loci_I  = ['A','B','C']
loci_II = ['DRB1','DQB1','DQA1','DPB1']

for sheet_name, loci in [('Class I (A, B, C)', loci_I),
                          ('Class II (DRB1, DQB1, DQA1, DPB1)', loci_II)]:
    ws = wb.active if sheet_name == 'Class I (A, B, C)' else wb.create_sheet(sheet_name)
    ws.title = sheet_name
    headers  = ['Patient','Group'] + \
               [c for l in loci for c in [f'{l}_1',f'{l}_1_prob',f'{l}_2',f'{l}_2_prob']] + \
               ['N alleles (all loci)','Signal']
    make_header_row(ws, headers)

    row_num = 2
    for grp in ['Healthy','CD','UC','CRC']:
        for _, r in genotype[genotype['Group']==grp].iterrows():
            signal   = 'No signal' if r['N'] == 0 else 'OK'
            row_data = [r['Patient'], r['Group']] + \
                       [r[c] for l in loci for c in [f'{l}_1',f'{l}_1_prob',f'{l}_2',f'{l}_2_prob']] + \
                       [r['N'], signal]
            write_row(ws, row_num, row_data,
                      fill_color=group_colors.get(grp,'FFFFFF'),
                      grey_italic=(signal=='No signal'))
            row_num += 1

    for i in range(1, len(headers)+1):
        ws.column_dimensions[get_column_letter(i)].width = 14 if i <= 2 else 13

# Summary sheet
ws3 = wb.create_sheet('Summary')
summary = [
    ['Rosati 2022 — HLA Predictions (HLAGuessr)', '', '', ''],
    [''],
    ['Method', 'HLAGuessr v0.1.13 (Ortega et al. 2025)'],
    ['Training set', 'Emerson + Russell + Rosati (1,039 donors)'],
    ['Alleles predicted', '94 modelable alleles'],
    ['QC threshold', f'\u2265{MIN_CLONES} clones per chain (TRA and TRB)'],
    [''],
    ['Clinical group', 'N total', 'N pass QC', 'N with signal'],
]
qc_stats = {
    'Healthy': (99, 96), 'CD': (114, 104), 'UC': (41, 33), 'CRC': (15, 3)
}
for grp, (n_tot, n_qc) in qc_stats.items():
    n_sig = len(genotype[(genotype['Group']==grp) & (genotype['N']>0)])
    summary.append([grp, n_tot, n_qc, n_sig])
summary += [
    [''],
    ['Mean alleles per patient (QC passed)', f"{genotype['N'].mean():.1f}"],
    ['Patients with no signal', f"{(genotype['N']==0).sum()} / {len(genotype)}"],
]
for i, row in enumerate(summary, 1):
    for j, val in enumerate(row, 1):
        ws3.cell(row=i, column=j, value=val).font = Font(name='Arial', size=10)
ws3['A1'].font = Font(name='Arial', size=13, bold=True)
ws3.column_dimensions['A'].width = 45
ws3.column_dimensions['B'].width = 20

OUT_XLSX = RESULTS_DIR + 'rosati_hla_predictions.xlsx'
wb.save(OUT_XLSX)
print(f'Saved: {OUT_XLSX}')
print(f'Sheets: {wb.sheetnames}')

## 9. Conclusions

- **269 patients** processed from 1,448 raw sequencing runs across four clinical groups (Healthy, CD, UC, CRC).
- **CRC repertoires** are near-empty (median 8 clones/patient) and uninformative for HLA inference; only 3/15 pass the depth QC.
- **233 patients pass QC** (≥1,000 clones per chain): 96 Healthy, 104 CD, 33 UC, 3 CRC.
- HLAGuessr produces **high-confidence predictions** (mean probability >0.98 when a carrier call is made) across all valid groups, consistent with Rosati's cohort having been part of the model's original training data (see caveat above).
- **19/233 patients (8.2%) show no signal** despite passing QC — their clonotype repertoires do not overlap with the 27,949 HLA-associated public clonotypes in the HLAGuessr training index.
- The mean predicted allele count is **6.2 per patient** across 7 loci (A, B, C, DRB1, DQB1, DQA1, DPB1), which is consistent with expected heterozygosity.
- **Ground-truth HLA genotypes for the CD and Healthy groups (192 patients) were subsequently obtained directly from Elisa Rosati** and used to validate these predictions — see `rosati_qc_hla_validation.ipynb` for the full validation and `README.md` for headline accuracy results.